---
title: "Tools, Evidence, and Domain Skills"
draft: true
categories: [agents, workflows, langgraph]
---


LangGraph controls transitions; it does not supply domain truth. Retrieval, extraction, citation, and source policy therefore have explicit contracts that can be tested without running a graph or a language model.

## Retrieval preserves exact locations

The fixture catalog contains supporting, contradictory, outdated, and irrelevant sources. Retrieval returns passages with offsets and records negative outcomes rather than collapsing them into an empty list.


In [1]:
from IPython.display import Markdown, display
from evidence_brief.adapters import ScriptedModelAdapter
from evidence_brief.domain import FixtureCatalog, retrieve
from evidence_brief.fixtures import load_corpus, request_for

model = ScriptedModelAdapter()
catalog = FixtureCatalog(load_corpus())
security_task = model.plan(request_for("conflict-01"))[0]
passages, observations = retrieve(catalog, security_task)

rows = "\n".join(
    f"| {p.source_id} | {p.start}:{p.end} | {p.text[:62]}… |" for p in passages
)
display(Markdown("| Source | Span | Passage |\n|---|---:|---|\n" + rows))
for passage in passages:
    source = catalog.sources[passage.source_id]
    assert source.text[passage.start:passage.end] == passage.text


| Source | Span | Passage |
|---|---:|---|
| vendor-security-2026 | 50:113 | Administrative actions are recorded in an exportable audit log… |
| vendor-security-2026 | 114:194 | The guide states that EU data residency is available for enter… |
| independent-audit-2026 | 0:51 | The audit verified encryption and audit-log export.… |
| independent-audit-2026 | 52:185 | It found that production indexes were stored only in the US re… |
| regulatory-policy-2026 | 0:176 | Production search services containing regulated documents requ… |

Offsets make a citation recoverable. The source ID alone can identify a document, but it cannot show which text supported the extracted claim.

## Evidence is not inference

The scripted adapter represents expected model behavior while remaining deterministic. It emits claims linked to passage IDs; the workflow can later permit an inference only when uncertainty is explicit.


In [2]:
from evidence_brief.domain import render_citation, verify_claim
from evidence_brief.schemas import Claim

claims = []
for passage in passages:
    claims.extend(model.extract(passage))
claims = list({claim.id: claim for claim in claims}.values())

for claim in claims:
    print(claim.kind, claim.text, render_citation(claim))
assert all(verify_claim(claim, passages) for claim in claims)

inference = Claim(
    id="residency-risk-inference",
    subject="adoption risk",
    predicate="recommendation",
    value="pilot_only",
    text="Unverified residency makes production adoption premature.",
    kind="inference",
    uncertainty="policy inference from conflicting evidence",
)
claims_with_inference = [*claims, inference]
without_offsets = [p.model_copy(update={"start": -1, "end": -1}) for p in passages]
collapsed_kinds = [c.model_copy(update={"kind": "evidence", "uncertainty": "none"}) for c in claims_with_inference]
print({
    "recoverable_citations": sum(p.start >= 0 for p in passages),
    "after_offset_ablation": sum(p.start >= 0 for p in without_offsets),
    "explicit_inference_labels": sum(c.kind == "inference" for c in claims_with_inference),
    "after_kind_ablation": sum(c.kind == "inference" for c in collapsed_kinds),
})
assert verify_claim(inference, passages)
assert sum(c.kind == "inference" for c in claims_with_inference) == 1
assert sum(c.kind == "inference" for c in collapsed_kinds) == 0


evidence AtlasVector exposes an exportable audit log. [vendor-security-2026#114-194]
evidence The vendor guide says EU residency is available. [vendor-security-2026#114-194]
evidence An independent audit verified audit-log export. [independent-audit-2026#52-185]
evidence The audit found that EU residency was not available. [independent-audit-2026#52-185]
evidence Regulated production requires verified residency. [regulatory-policy-2026#0-176]
{'recoverable_citations': 5, 'after_offset_ablation': 0, 'explicit_inference_labels': 1, 'after_kind_ablation': 0}


The graph will carry these objects unchanged. Skills may alter query formulation or source-quality guidance, but the citation invariant lives in deterministic validation, where a prompt change cannot silently remove it.
